### Objective: Demonstrate that we can produce a temporally smooth, daily, representation of surface water.

### Completed (not tested) steps:
##### Step d1. establish the timeframe of interest and spatial analysis frame. Bilinear resampling.
##### Step d2. gather MSI and OLI data
- relevant bands only (ok),
- rename variables as needed (ok),
- group by day (ok),
- rescale to sr * 10,000 (ok),
- rescale to kelvin (ok)
- merge datasets (ok) 
##### Step d3. gather the annual WOfS data
##### Step d4. get the geomedians as references for data cleaning.

### Incomplete steps:
#### Data
##### Step d2a. apply corrections?
##### Step d4. gather the wofls data.

#### Data cleaning
##### 

#### Analysis
##### Step a1. produce a 'gappy' NDWI from MSI and OLI data
##### Step a2. extract the coincident NDWI values and regress against each other.  
##### Step a3. adjust the OLI NDWI values to match MSI derived NDWI
##### Step a4. compare the adjusted NDWI values with clear wofls and establish the best break point for water / non-water. In the crop monitoring workbook there is an algorithm used in this way that it might be possible to use in this context also.
##### Step a5. interpolate the NDWI values through time, and replace null values with interpolated values

#### Results
##### Step r1. Plot the thresholded NDWI through time. This becomes the daily estimated area of water. 
##### Step r2. Use the daily water extent as the extent for WQ monitoring.
##### Step r3. If it works, use the daily water extent as the basis for a flood monitoring workbook (enhance with Sentinel-1).




#### Load packages
There are other packages that should be loaded but somewhat obscure (DE Afrcia)

In [6]:
import datacube
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from deafrica_tools.plotting import rgb
from deafrica_tools.app import geomedian

#### Set the space-time domain

In [7]:
#Set the space-time domain(s) of interest using a python dictionary object:
year1 = "2018-01-01"
year2 = "2018-01-31"

#few_pixels is useful for debugging and exploration
Weija_Reservoir  = [{"x": (-0.325, -0.41),     "y": ( 5.35, 6.20),       "time": (year1,year2)},""]
Senegal_StLouis  = [{"x": (-15.74,-15.84),     "y": (16.3, 16.3900),     "time": (year1,year2)},"Lac de Guiers"]
Lake_Sulunga     = [{"x": (34.95, 35.4),       "y": (-6.3, -5.8),        "time": (year1,year2)},""]
#this is a higly variable waterbody. annual wofs will be needed to work with this. 
         
Lake_Victoria_NE = [{'x': (33.5,34.8),     'y': (-.6,0.4),   'time':(year1,year2)},"Lake Victoria NE"]

#Manyara is a shallow alkaline lake 10 feet deep. https://wildlifesafaritanzania.com/facts-about-lake-manyara-national-park/
#During sampling periods[2006-2008], big blooms of blue-green algae were observed covering thewhole 
#water surface especially in March, April, May and June. 
#Most of the blooms were along the lake shore, giving a characteristic foul smell and foam 
#https://www.researchgate.net/publication/266095025_Assessment_of_farming_practices_and_uses_of_agrochemicals_in_Lake_Manyara_basin_Tanzania
#(6) (PDF) Assessment of farming practices and uses of agrochemicals in Lake Manyara basin, Tanzania. Available from: https://www.researchgate.net/publication/266095025_Assessment_of_farming_practices_and_uses_of_agrochemicals_in_Lake_Manyara_basin_Tanzania [accessed Sep 07 2024].

few_pixels         = [{"x": (33.1600,33.16005),   "y": (-2.1200, -2.1424),  "time": (year1,year2)},""]
small_area         = [{"x": (33.1655,33.1864),    "y": (-2.1444,-2.1532),   "time": (year1,year2)},""]
cameroon_waterbody = [{"y": (6.20,6.30),          "x": (11.25, 11.35),      "time": (year1,year2)},""]
Lake_Victoria      = [{"x": (33.100,33.300),      "y": (-2.0800,-2.1500),   "time": (year1,year2)},""]
Lake_Mweru         = [{"x": (28.200,29.300),      "y": (-8.500,-9.500),     "time": (year1,year2)},""]
Ghana_AwunaBeach   = [{"x": (-1.580,-1.640),      "y": (  5.0, 5.05),       "time": (year1,year2)},""]

Large_area         = [{"x": ( 32.5,   35.5),      "y" : ( -4.5,-1.5),       "time": (year1,year2)},""]
Lake_vic_west      = [{"x": ( 32.5, 32.78),       "y" : ( -2.65,-2.3),      "time": (year1,year2)},""]
Lake_vic_east      = [{"x": ( 32.78, 33.3),       "y" : ( -2.65,-2.3),      "time": (year1,year2)},""]
Morocco            = [{"x": (-7.45, -7.65),       "y" : (  32.4,32.5),      "time": (year1,year2)},"Barrage Al Massira"]

Thewaterskool_SA   = [{"x": (19.1, 19.3),         "y" : ( -34.1  , -33.98), "time": (year1,year2)},""]
SA_dam             = [{"x": ( 19.35,   19.47),    "y" : ( -33.800, -33.650),"time": (year1,year2)},""]
Ethiopia1of2       = [{"x": ( 38.35,   38.65),    "y" : (   7.37 ,   7.55), "time": (year1,year2)},"Ethiopia, Lake Abiata 1"]
Ethiopia2of2       = [{"x": ( 38.66,   38.83),    "y" : (   7.50 ,   7.71), "time": (year1,year2)},"Ethiopia, Lake Abiata 2"]
Ethiopia3of2       = [{"x": ( 38.50,   38.67),    "y" : (   7.55 ,   7.69), "time": (year1,year2)},"Ethiopia, Lake Abiata"]
Ethiopia_Lake_Tana = [{"x": ( 37.05,   37.22),    "y" : (  11.9  ,  12.0),  "time": (year1,year2)},"Ethiopia_Lake_Tana"]
Mare_aux_Vacoas    = [{"x": ( 57.485,  57.524),   "y" : ( -20.389, -20.359),"time": (year1,year2)},"Mare_aux_Vacoas"]
SA_smalldam        = [{"x": ( 19.494,  19.498),   "y" : ( -33.802, -33.800),"time": (year1,year2)},"Irrigation Dam, South Africa"]
Ethiopia_both      = [{"x": ( 38.35,   38.83),    "y" : (   7.37 ,   7.71), "time": (year1,year2)},"Ethiopia, Lake Abiata +"]
Madagascar         = [{"x": ( 43.58 ,  43.76 ),   "y" : ( -22.03 , -21.87 ),"time": (year1,year2)},"Farihy Ihotry, Madagascar"]
Lake_Manyara       = [{"x": ( 35.68 ,  35.92 ),   "y" : ( -03.86 , -03.38), "time": ('1999',year2)},"Lake_Manyara, Tanzania"] #this is the lake to use as an example of monitoring, see 2015-12-28
Farihy_         = [{"x": ( 43.58 ,  43.76 ),   "y" : ( -22.03 , -21.87 ),"time": (year1,year2)},"Farihy Ihotry, Madagascar"]
Farihy_itasy      = [{"x": ( 46.73 ,  46.83 ),   "y" : ( -19.10 , -19.04 ),"time": ('2015','2023')},"Farihy Itasy, Madagascar"]
Kolokonda         = [{"x": ( 35.4888, 35.5488),   "y" : ( -5.976, -5.916 ),"time": ('1991','2021')},"Dodoma, Tanzanier"]
Dodoma_small      = [{"x": ( 35.475 , 35.51),   "y" : ( -6.03, -5.99 ),"time": ('1991','2021')},"Dodoma, Tanzanier"]
size_test         = [{"x": ( 31.400 , 32.40),   "y" : ( -0.00, -1.00 ),"time": (year1,year2)},"Lake Victoria"]
lake_vic_all      = [{"x": ( 31.500 , 34.86),   "y" : ( -3.00, +0.50 ),"time": (year1,year2)},"Lake Victoria"]


#only the last one will matter here, of course
#spacetime_domain = eye_of_africa
#spacetime_domain = few_pixels
#spacetime_domain = small_area

spacetime_domain = Lake_vic_east
spacetime_domain = Lake_vic_west
spacetime_domain = Lake_Victoria
spacetime_domain = Ghana_AwunaBeach
spacetime_domain = Lake_Victoria_NE
spacetime_domain = Lake_Manyara
spacetime_domain = SA_dam
spacetime_domain = Ethiopia_Lake_Tana
spacetime_domain = Ghana_AwunaBeach
spacetime_domain = Lake_Sulunga
spacetime_domain = Mare_aux_Vacoas
spacetime_domain = Ethiopia3of2
spacetime_domain = Ghana_AwunaBeach
spacetime_domain = Lake_Mweru
spacetime_domain = Mare_aux_Vacoas
spacetime_domain = Thewaterskool_SA
spacetime_domain = cameroon_waterbody
spacetime_domain = Large_area
spacetime_domain = Morocco
spacetime_domain = Ethiopia_both
spacetime_domain = Ethiopia_both
spacetime_domain = Ethiopia1of2
spacetime_domain = Madagascar
spacetime_domain = Weija_Reservoir
spacetime_domain = Lake_Manyara
spacetime_domain = Senegal_StLouis
spacetime_domain = Morocco
spacetime_domain = Lake_Manyara
spacetime_domain = Farihy_itasy
spacetime_domain = Dodoma_small
spacetime_domain = Kolokonda
spacetime_domain = lake_vic_all
spacetime_domain = SA_smalldam

file_name = spacetime_domain
site_name = spacetime_domain[1]
spacetime_domain = spacetime_domain[0]

#establish a reasonable grid resolution between min and max based on the AOI
cell_min = 10
cell_max = 500
x0 = spacetime_domain["x"][0]
x1 = spacetime_domain["x"][1]
y0 = spacetime_domain["y"][0]
y1 = spacetime_domain["y"][1]
dxm = ((x1-x0) * 100000) * np.cos(y0 * np.pi/180.0) #metres approx allowing for lattitudinal compression of dx
dym = ((y1-y0) * 100000) #metres approx
dAm = dxm * dym        #sqare metres total extent
max_cells = 1000000  #grid cells covering the extent, about 100km2 at 10m resolution
cell_m2 = dAm / max_cells
cell_dxm = abs(cell_m2)**0.5


if cell_dxm < cell_min :
    cell_dxm = cell_min
else :
    if cell_dxm > cell_max :
        cell_dxm = cell_max
cell_dxm = int(cell_dxm / 10) * 10
grid_resolution = (-cell_dxm,cell_dxm) 
cellcount = dAm / (cell_dxm**2)

aspect_ratio = np.abs(dym/dxm)
cell_area = (cell_dxm**2)/1000000 #km^2

#apply a bilinear interpolation if cells that are small relative to the pixels
if cell_dxm > 60 :
    resampling_option = "nearest"
else:
    resampling_option = "bilinear"

year1 = spacetime_domain["time"][0]
year2 = spacetime_domain["time"][1]
    
print("grid resolution will be:", grid_resolution)
print("rough dimensions (x,y): ", int(dxm/1000), " by " ,int(abs(dym/1000)), "kilometres")
print("total cells is roughly: ", int(cellcount))
print("cell area is: ", cell_area," km2")
print("resampling :",resampling_option)
print("site name: ", site_name)
print("years:" , year1, year2)


grid resolution will be: (-10, 10)
rough dimensions (x,y):  0  by  0 kilometres
total cells is roughly:  664
cell area is:  0.0001  km2
resampling : bilinear
site name:  Irrigation Dam, South Africa
years: 2018-01-01 2018-12-31


#### Connect to the datacube

In [8]:
dc = datacube.Datacube(app='pointless?') #testing what difference the 'app' value makes...

In [9]:
product_name = 'gm_s2_annual'
product_name = 'wofs_ls_summary_alltime'
product_name = 'ls8_sr'

product_name = "wofs_ls_summary_annual"
product_name = 's2_l2a'
product_name = 'ls8_st'
dc_measurements = dc.list_measurements()
dc_measurements.loc[product_name].drop('flags_definition', axis=1)

,name,dtype,units,nodata,aliases
measurement,,,,,
ST_B10,ST_B10,uint16,Kelvin,0.0,"[band_10, st, surface_temperature]"
ST_TRAD,ST_TRAD,int16,W/(m2.sr.μm),-9999.0,"[trad, thermal_radiance]"
ST_URAD,ST_URAD,int16,W/(m2.sr.μm),-9999.0,"[urad, upwell_radiance]"
ST_DRAD,ST_DRAD,int16,W/(m2.sr.μm),-9999.0,"[drad, downwell_radiance]"
ST_ATRAN,ST_ATRAN,int16,1,-9999.0,"[atran, atmospheric_transmittance]"
ST_EMIS,ST_EMIS,int16,1,-9999.0,"[emis, emissivity]"
ST_EMSD,ST_EMSD,int16,1,-9999.0,"[emsd, emissivity_stddev]"
ST_CDIST,ST_CDIST,int16,Kilometers,-9999.0,"[cdist, cloud_distance]"
QA_PIXEL,QA_PIXEL,uint16,bit_index,1.0,"[pq, pixel_quality]"


#### Set the measurements to be used ('bands')

In [10]:
#set the measurement (variables) domain using a python dictionary object:

mmts_wofs = {"measurements":("frequency","count_clear","count_wet")}
mmts_msi  = {"measurements":(
    "B01","B02","B03","B04",     #need all these to be able to compare with geomedian
    "B05","B06","B07",
    #"B08","B8A","B09","B11",
    "B12",
    #"SCL","AOT","WVP"
    )}
mmts_oli  = {"measurements":(
    "SR_B1","SR_B2","SR_B3","SR_B4",
    #"SR_B5","SR_B6","SR_B7",
    "QA_PIXEL","QA_RADSAT"
    )}
mmts_oli_geom = {"measurements":(
    "SR_B2","SR_B3","SR_B4",
    #"SR_B5","SR_B6","SR_B7",
    "count", #"smad","emad","bcmad",
    )}
mmts_msi_geom = {"measurements":("B02","B03","B04","B12","B05","B06","B07","count",
    "smad","count",#"emad","bcmad",
    )}       
mmts_tirs = {"measurements":(
    "ST_B10",  # 	ST_B10 	uint16 	Kelvin 	0.0 	[band_10, st, surface_temperature]
    #"ST_TRAD", # 	ST_TRAD 	int16 	W/(m2.sr.μm) 	-9999.0 	[trad, thermal_radiance]
    #"ST_URAD", # 	ST_URAD 	int16 	W/(m2.sr.μm) 	-9999.0 	[urad, upwell_radiance]
    #"ST_DRAD", # 	ST_DRAD 	int16 	W/(m2.sr.μm) 	-9999.0 	[drad, downwell_radiance]
    #"ST_ATRAN", # 	ST_ATRAN 	int16 	1 	-9999.0 	[atran, atmospheric_transmittance]
    "ST_EMIS", # 	ST_EMIS 	int16 	1 	-9999.0 	[emis, emissivity]
    "ST_EMSD", # 	ST_EMSD 	int16 	1 	-9999.0 	[emsd, emissivity_stddev]
    #"ST_CDIST", # 	ST_CDIST 	int16 	Kilometers 	-9999.0 	[cdist, cloud_distance]
    #"QA_PIXEL", # 	QA_PIXEL 	uint16 	bit_index 	1.0 	[pq, pixel_quality]
    #"QA_RADSAT", # 	QA_RADSAT 	uint16 	bit_index 	0.0 	[radsat, radiometric_saturation]
    "ST_QA",     # 	ST_QA 	int16 	Kelvin 	-9999.0 	[st_qa, surface_temperature_quality]
    )} 

### Load the data from the datacube for the **space**, **time** and **measurement** domains of interest

In [68]:
# decide which instruments are to be used
_oli = True
_msi = True
_wofs = True
_tirs = True

In [42]:
#helpful function
def rename_vars_robust(dataset,var_names):
        for var in var_names :
            if var[0] in np.asarray(dataset.data_vars):
                dataset = dataset.rename_vars({var[0]:var[1]})
        return(dataset)

In [43]:
# load annual wofs data
if _wofs:
    wofs   = dc.load(
    product="wofs_ls_summary_annual",
    **mmts_wofs,
    **spacetime_domain,
    resolution=grid_resolution,
    resampling=resampling_option,
    align=(0,0),     
    ).squeeze()
#  like=ds.geobox).filtered.squeeze()

    #give the variables logical and unique names:
    wofs_var_names =(
                    ("frequency","wofs_freq"),
                    ("count_clear","wofs_clearcount"),
                    ("count_wet","wofs_wetcount"),
                    )
    wofs = rename_vars_robust(wofs,wofs_var_names)

In [69]:
# load oli data with parameters previously set
if _oli: 
    oli  = dc.load(product=("ls8_sr","ls9_sr"),
             **spacetime_domain,
             **mmts_oli,
             resolution=grid_resolution,
             align=(0,0),
             resampling=resampling_option,
             output_crs='EPSG:6933',
             group_by="solar_day")  
 
    #re-name the variables:
    oli_var_names = (
                    ("SR_B1","oli01"),
                    ("SR_B2","oli02"),
                    ("SR_B3","oli03"),
                    ("SR_B4","oli04"),
                    ("SR_B5","oli05"),
                    ("SR_B6","oli06"),
                    ("SR_B7","oli07"),
                    ("QA_PIXEL","oliqa_pixel"),
                    ("QA_RADSAT","oliqa_radsat"),
                    )
    oli = rename_vars_robust(oli,oli_var_names) 

    #re-scale to SR 0-10,000

    oli_rescale_vars = ("oli01","oli02","oli03","oli04","oli05","oli06","oli07") 
    for var in oli_rescale_vars:
        if var in np.asarray(oli.data_vars):
            oli[var] =  ((2.75e-5 * oli[var]) - 0.2 ) * 10000

In [71]:
# load tirs data with parameters previously set
if _tirs: 
    tirs  = dc.load(product=("ls8_st","ls9_st"),
             **spacetime_domain,
             **mmts_tirs,
             resolution=grid_resolution,
             align=(0,0),
             resampling=resampling_option,
             output_crs='EPSG:6933',
             group_by="solar_day",
             )
     #rename variables
    tirs_var_names= (
                    ("ST_B10","tirs_kelvin"),  # 	ST_B10 	uint16 	Kelvin 	0.0 	[band_10, st, surface_temperature]
                    ("ST_TRAD","tirs_trad"),   # 	ST_TRAD 	int16 	W/(m2.sr.μm) 	-9999.0 	[trad, thermal_radiance]
                    ("ST_URAD","tirs_urad"),   # 	ST_URAD 	int16 	W/(m2.sr.μm) 	-9999.0 	[urad, upwell_radiance]
                    ("ST_DRAD","tirs_drad"),   # 	ST_DRAD 	int16 	W/(m2.sr.μm) 	-9999.0 	[drad, downwell_radiance]
                    ("ST_ATRAN","tirs_atran"),  # 	ST_ATRAN 	int16 	1 	-9999.0 	[atran, atmospheric_transmittance]
                    ("ST_EMIS","tirs_emis"),   # 	ST_EMIS 	int16 	1 	-9999.0 	[emis, emissivity]
                    ("ST_EMSD","tirs_emsd"),   # 	ST_EMSD 	int16 	1 	-9999.0 	[emsd, emissivity_stddev]
                    ("ST_CDIST","tirs_cdist"), # 	ST_CDIST 	int16 	Kilometers 	-9999.0 	[cdist, cloud_distance]
                    ("QA_PIXEL","tirs_qa_pixel"), # 	QA_PIXEL 	uint16 	bit_index 	1.0 	[pq, pixel_quality]
                    ("QA_RADSAT","tirs_qa_radsat"), # 	QA_RADSAT 	uint16 	bit_index 	0.0 	[radsat, radiometric_saturation]
                    ("ST_QA","tirs_qa")     # 	ST_QA 	int16 	Kelvin 	-9999.0 	[st_qa, surface_temperature_quality]
                    )

    tirs = rename_vars_robust(tirs,tirs_var_names)

    #re-scale data to kelvin (not sure if we also need to scale the sd and other variables, such as emissivity)
    tirs.tirs_kelvin = tirs.tirs_kelvin * 0.00341802 + 149.0

In [ ]:
# load msi data with parameters previously set
if _msi: 
    msi  = dc.load(product=("s2_l2a"),
             **spacetime_domain,
             **mmts_msi,
             resolution=grid_resolution,
             align=(0,0),
             resampling=resampling_option,
             output_crs='EPSG:6933',
             group_by="solar_day"  
             )
    

    msi_var_names =      (
                     ("B01","msi01"),
                     ("B02","msi02"),
                     ("B03","msi03"),
                     ("B04","msi04"),
                     ("B05","msi05"),
                     ("B06","msi06"),
                     ("B07","msi07"),
                     ("B08","msi08"),
                     ("B8A","msi08a"),
                     ("B09","msi09"),
                     ("B11","msi11"),
                     ("B12","msi12"),
                     ("SCL","msi_scl"),
                     ("AOT","msi_aot"),
                     ("WVP","msi_wvp"),
                     )

    msi = rename_vars_robust(msi,msi_var_names)

In [ ]:
# load oli geomedian, ideally from before and after
if _oli_geom: 
    geomedian  = dc.load(product=("gm"),
             **spacetime_domain,
             **mmts_msi,
             resolution=grid_resolution,
             align=(0,0),
             resampling=resampling_option,
             output_crs='EPSG:6933',
             group_by="solar_day"  
             )
    

    msi_var_names =      (
                     ("B01","msi01"),
                     ("B02","msi02"),
                     ("B03","msi03"),
                     ("B04","msi04"),
                     ("B05","msi05"),
                     ("B06","msi06"),
                     ("B07","msi07"),
                     ("B08","msi08"),
                     ("B8A","msi08a"),
                     ("B09","msi09"),
                     ("B11","msi11"),
                     ("B12","msi12"),
                     ("SCL","msi_scl"),
                     ("AOT","msi_aot"),
                     ("WVP","msi_wvp"),
                     )

    msi = rename_vars_robust(msi,msi_var_names)

In [87]:
# merge the datasets to single series
# this merge appears to work, but assumes all instruments are used
ds = xr.merge([oli,msi,tirs,wofs])

Now plot the `smad` band to see if it is useful for distinguishing between irrigated and non-irrigated fields

In [ ]:
fig,ax=plt.subplots(1,2, sharey=True, figsize=(15,7))
rgb(ds,ax=ax[0])
ds.smad.plot.imshow(vmax=0.025, ax=ax[1])
ax[0].set_title('True Colour')
ax[1].set_title('Spectral MAD');

####  Use WOfS to find the best threshold for NDWI.

This process uses scikit for image processing in python. 

It looks like `smad` will be useful for identifying irrigation, so lets define a threshold using the `skimage.filters.threshold_li` method. This method will automatically find the optimal threhold based on minimizing the cross-entropy between the foreground and the foreground mean, and the background and the background mean

In [ ]:
from skimage.filters import threshold_li

#find the threshold
threshold = threshold_li(ds.smad.values)
print("Threshold: ", threshold)

#plot a histogram of the result
fig, ax = plt.subplots(figsize=(10, 3))
ds.smad.plot.hist(bins=1000, label="SMAD")
plt.xlim(0,0.003)
ax.axvspan(xmin=0, xmax=threshold, alpha=0.25, color="green", label="Non-irrigated"),
ax.axvspan(xmin=threshold,
           xmax=0.005,
           alpha=0.25,
           color="red",
           label="Irrigated")
plt.legend();

### Apply the threshold and plot the result

In [ ]:
#Apply the threshold
irrigated = ds.smad >= threshold

#plot
fig,ax=plt.subplots(1,2, sharey=True, figsize=(15,7))
rgb(ds,ax=ax[0])
irrigated.plot.imshow(ax=ax[1], add_colorbar=False)
ax[0].set_title('True Colour')
ax[1].set_title('Irrigated areas');

***

## Additional information

**License:** The code in this notebook is licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0). 
Digital Earth Africa data is licensed under the [Creative Commons by Attribution 4.0](https://creativecommons.org/licenses/by/4.0/) license.

**Contact:** If you need assistance, please post a question on the [Open Data Cube Slack channel](http://slack.opendatacube.org/) or on the [GIS Stack Exchange](https://gis.stackexchange.com/questions/ask?tags=open-data-cube) using the `open-data-cube` tag (you can view previously asked questions [here](https://gis.stackexchange.com/questions/tagged/open-data-cube)).
If you would like to report an issue with this notebook, you can file one on [Github](https://github.com/digitalearthafrica/deafrica-sandbox-notebooks).

**Compatible datacube version:** 

In [ ]:
print(datacube.__version__)

**Last Tested:**

In [ ]:
from datetime import datetime
datetime.today().strftime('%Y-%m-%d')